# 🎙️ Voice Cloning — Bangla + English (Colab)

Nijer voice sample diye je kono text (Bangla ba English) cloned voice te bolabe.

**Age koro:** `Runtime` → `Change runtime type` → **T4 GPU** select koro, tarpor Save.

Cell gulo upor theke niche ek ek kore run koro (▶ button ba `Shift+Enter`).

| Section | Ki kore |
|---|---|
| 1 | GPU check |
| 2 | Reference voice upload |
| 3 | (optional) Noise remove |
| 4 | **English** clone — XTTS-v2 |
| 5 | **Bangla** clone — Chatterbox-Bangla (reference audio diye) |
| 6 | **Fine-tune** (5-min, highest quality) — GPT-SoVITS |

## 1. GPU check
T4/other GPU dekhale thik ache. `cpu` dekhale runtime type change koro (upore dekho).

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️  GPU nai! Runtime > Change runtime type > T4 GPU koro.')
!nvidia-smi -L

## 2. Reference voice upload
Tomar clean recording (`sample.wav`, 10-30 sec zero-shot er jonno enough) upload koro.
**Ei same REF diyei Bangla o English dutai clone hobe.**

In [ ]:
from google.colab import files
import os
os.makedirs('/content/voices', exist_ok=True)
os.makedirs('/content/output', exist_ok=True)
print('sample.wav (ba mp3) upload koro...')
up = files.upload()
REF = '/content/voices/' + list(up.keys())[0]
os.replace(list(up.keys())[0], REF)
print('Reference set:', REF)

## 3. (Optional) Noise remove
Recording e background noise thakle eta run koro. Clean thakle skip koro.

In [ ]:
!pip -q install noisereduce soundfile librosa
import noisereduce as nr, soundfile as sf, librosa
y, sr = librosa.load(REF, sr=None, mono=True)
y_clean = nr.reduce_noise(y=y, sr=sr)
REF = '/content/voices/clean.wav'
sf.write(REF, y_clean, sr)
print('Denoised ->', REF)
from IPython.display import Audio; Audio(REF)

## 4. 🇬🇧 English clone — XTTS-v2
English (+ 16 lang) er jonno best. Prothombar model download hobe (~1.8GB).

In [ ]:
!pip -q install TTS==0.22.0
import os
os.environ['COQUI_TOS_AGREED'] = '1'
from TTS.api import TTS
xtts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to('cuda')
print('XTTS ready ✓')

In [ ]:
# ==== EKHANE TOMAR TEXT LIKHO ====
TEXT_EN = "Hello, this is my cloned voice. Today I will read whatever text you give me."
LANG = 'en'   # en, hi, ar, es, fr, de, it, pt, ru, zh-cn, ja, ko ...
# =================================
out = '/content/output/english.wav'
xtts.tts_to_file(text=TEXT_EN, speaker_wav=REF, language=LANG, file_path=out)
print('Saved:', out)
from IPython.display import Audio; Audio(out)

## 5. 🇧🇩 Bangla clone — Chatterbox-Bangla (reference audio diye)
[Banglabox/chatterbox-bangla-tts](https://huggingface.co/Banglabox/chatterbox-bangla-tts) — **zero-shot Bangla voice clone.**
Tomar REF (Section 2) diye Bangla te bolabe.

**5a — model + code download:**

In [ ]:
!pip -q install chatterbox-tts huggingface_hub soundfile
from huggingface_hub import snapshot_download
import os
BANGLA_DIR = snapshot_download(repo_id='Banglabox/chatterbox-bangla-tts')
print('Downloaded ->', BANGLA_DIR)

# repo er requirements thakle install
req = os.path.join(BANGLA_DIR, 'requirements.txt')
if os.path.exists(req):
    !pip -q install -r "{req}"

# infer.py kothay ache locate kori (inference/ ba root — version onujayi)
def _find(root, name):
    for d, _, fs in os.walk(root):
        if name in fs:
            return d
    return None
INFER_DIR = _find(BANGLA_DIR, 'infer.py')
print('infer.py folder ->', INFER_DIR)
assert INFER_DIR, 'infer.py pawa jay nai — repo structure bodleche hote pare.'

**5b — generate.** infer.py nijer folder theke run kori (author er default path structure thik thake), `--out` diye output path fix kori, tarpor auto-play.

In [ ]:
# ==== EKHANE BANGLA TEXT LIKHO ====
TEXT_BN = "নমস্কার, এটা আমার ক্লোন করা কণ্ঠস্বর। আপনি যে লেখা দেবেন আমি সেটাই পড়ে শোনাবো।"
# ==================================
import os
REF_ABS = os.path.abspath(REF)
OUT = '/content/output/bangla.wav'

# (prothombar flag confirm korte chaile:  !cd "{INFER_DIR}" && python infer.py --help )
!cd "{INFER_DIR}" && python infer.py --text "{TEXT_BN}" --ref "{REF_ABS}" --out "{OUT}"

from IPython.display import Audio, display
if os.path.exists(OUT):
    print('Saved:', OUT)
    display(Audio(OUT))
else:
    print('⚠️ Output toiri hoy nai. Uporer log dekho — flag ba dependency (peft) issue hote pare.')
    print('   Flag confirm korte:  !cd "' + str(INFER_DIR) + '" && python infer.py --help')

> **Note:** eta specifically Bangla er jonno fine-tuned (adapter + Bangla tokenizer), tai Bangla quality XTTS er cheye onek bhalo. English er jonno Section 4 (XTTS) use koro. Same REF dile dutai eki voice.

## 6. 🏆 Fine-tune (highest quality) — GPT-SoVITS
Tomar **5-min recording** diye nijer model train korte chaile — dedicated notebook ache:
**`gpt_sovits_train_colab.ipynb`** (slice → ASR → train → inference, WebUI shoho).

- Repo: https://github.com/RVC-Boss/GPT-SoVITS

## ✅ Result download
Generated wav gulo `/content/output/` folder e. Download:

In [ ]:
from google.colab import files
import os
for f in os.listdir('/content/output'):
    print('downloading', f)
    files.download('/content/output/' + f)